# Notebook 3: Normalizing Flows from Scratch

In this notebook, we will implement a **Normalizing Flow** from scratch using PyTorch. Normalizing flows are a powerful class of generative models that allow for exact density estimation and sampling.

## Concept

A normalizing flow transforms a simple base distribution $p_z(z)$ (usually a standard Gaussian) into a complex target distribution $p_x(x)$ using a sequence of invertible transformations $f$.

$$ x = f(z) $$
$$ z = f^{-1}(x) $$

### Change of Variables Formula

The probability density of the transformed variable $x$ is given by:

$$ \log p_x(x) = \log p_z(z) - \log |\det J_f(z)| $$

Where $J_f(z)$ is the Jacobian of the transformation $f$.

## RealNVP (Real Non-Volume Preserving)

We will implement the **Affine Coupling Layer**, a key building block of RealNVP. It splits the input $x$ into two parts $(x_a, x_b)$ and transforms $x_b$ based on $x_a$.

$$ y_a = x_a $$
$$ y_b = x_b \odot \exp(s(x_a)) + t(x_a) $$

Here, $s$ (scale) and $t$ (translation) are neural networks. This transformation is easily invertible and its Jacobian determinant is simple to compute (it's triangular).

## Task: Density Estimation on 'Two Moons'

We will learn the distribution of the classic 'Two Moons' dataset.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.datasets import make_moons
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)

# 1. Generate Data
def get_data(n_samples=2000):
    X, _ = make_moons(n_samples=n_samples, noise=0.05)
    return torch.tensor(X, dtype=torch.float32)

X_train = get_data()

plt.figure(figsize=(6, 6))
plt.scatter(X_train[:, 0], X_train[:, 1], s=5)
plt.title('Target Distribution: Two Moons')
plt.show()

## Implementing the Affine Coupling Layer

We need a neural network to predict the scale $s$ and translation $t$ parameters.

In [ ]:
class CouplingLayer(nn.Module):
    def __init__(self, input_dim, hidden_dim, mask):
        super().__init__()
        self.mask = mask
        # Neural network to predict s and t
        # Input is masked input, output is concatenation of s and t
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.Tanh(), # Tanh is often more stable for flows
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, input_dim * 2) # Output s and t
        )
    
    def forward(self, x):
        # x: [batch, input_dim]
        
        # Mask the input: x_a stays constant, x_b is transformed
        # If mask is [1, 0], x * mask gives [x1, 0]
        x_masked = x * self.mask
        
        # Predict s and t based on x_masked
        st = self.net(x_masked)
        s, t = st.chunk(2, dim=1)
        
        # We only apply transformation to the unmasked part.
        # To ensure stability, we often use tanh or clamp for s
        s = torch.tanh(s) 
        
        # Apply affine transformation:
        # y = x * exp(s) + t
        # But we only want to apply this where mask is 0.
        # Where mask is 1, y = x.
        
        # Scaling factor: exp(s) * (1 - mask) + mask
        # If mask=1: factor = 1. If mask=0: factor = exp(s)
        scale = torch.exp(s) * (1 - self.mask) + self.mask
        
        # Translation: t * (1 - mask)
        translation = t * (1 - self.mask)
        
        y = x * scale + translation
        
        # Log Determinant of Jacobian
        # Since it's diagonal (or triangular), it's sum of log diagonal entries.
        # log_det = sum(log(scale)) = sum(s * (1 - mask))
        log_det = torch.sum(s * (1 - self.mask), dim=1)
        
        return y, log_det
    
    def inverse(self, y):
        # Inverse transformation
        # x = (y - t) * exp(-s)
        
        y_masked = y * self.mask
        st = self.net(y_masked)
        s, t = st.chunk(2, dim=1)
        s = torch.tanh(s)
        
        scale = torch.exp(s) * (1 - self.mask) + self.mask
        translation = t * (1 - self.mask)
        
        x = (y - translation) / scale
        return x

## Building the RealNVP Model

A single coupling layer only transforms half the variables. To transform all variables, we stack multiple layers and alternate the masks.

In [ ]:
class RealNVP(nn.Module):
    def __init__(self, input_dim, hidden_dim, n_layers=4):
        super().__init__()
        self.layers = nn.ModuleList()
        
        # Alternate masks: [1, 0] then [0, 1]
        mask = torch.tensor([1, 0], dtype=torch.float32)
        
        for i in range(n_layers):
            self.layers.append(CouplingLayer(input_dim, hidden_dim, mask))
            mask = 1 - mask # Flip mask
            
    def forward(self, x):
        log_det_sum = 0
        for layer in self.layers:
            x, log_det = layer(x)
            log_det_sum += log_det
        return x, log_det_sum
    
    def inverse(self, z):
        for layer in reversed(self.layers):
            z = layer.inverse(z)
        return z

## Training

We minimize the **Negative Log Likelihood (NLL)**.

$$ \mathcal{L} = - \mathbb{E}_{x \sim p_{data}} [\log p_x(x)] $$

Using the change of variables:
$$ \log p_x(x) = \log p_z(z) + \log |\det J| $$
$$ \mathcal{L} = - \mathbb{E} [ \log p_z(f(x)) + \sum \log |\det J_i| ] $$

In [ ]:
input_dim = 2
hidden_dim = 64
flow = RealNVP(input_dim, hidden_dim, n_layers=6)
optimizer = optim.Adam(flow.parameters(), lr=0.001)

# Standard Normal Prior
prior = torch.distributions.MultivariateNormal(torch.zeros(2), torch.eye(2))

losses = []
for epoch in range(500):
    optimizer.zero_grad()
    
    # Forward pass: x -> z
    z, log_det = flow(X_train)
    
    # Calculate Loss: - (log_prob_prior(z) + log_det)
    log_prob_z = prior.log_prob(z)
    loss = -torch.mean(log_prob_z + log_det)
    
    loss.backward()
    optimizer.step()
    
    losses.append(loss.item())
    if epoch % 100 == 0:
        print(f'Epoch {epoch}, Loss: {loss.item():.4f}')

plt.plot(losses)
plt.title('NLL Loss')
plt.show()

## Visualization

1.  **Latent Space**: Transform data $x$ to $z$. Should look Gaussian.
2.  **Sampling**: Sample $z \sim \mathcal{N}(0, I)$, transform to $x$. Should look like moons.

In [ ]:
# 1. Latent Space
with torch.no_grad():
    z_transformed, _ = flow(X_train)
    z_transformed = z_transformed.numpy()

# 2. Sampling
with torch.no_grad():
    z_sample = prior.sample((2000,))
    x_generated = flow.inverse(z_sample).numpy()

fig, axs = plt.subplots(1, 3, figsize=(15, 5))

axs[0].scatter(X_train[:, 0], X_train[:, 1], s=5)
axs[0].set_title('Original Data (X)')

axs[1].scatter(z_transformed[:, 0], z_transformed[:, 1], s=5, color='orange')
axs[1].set_title('Transformed to Latent Space (Z)')
axs[1].set_xlim(-4, 4)
axs[1].set_ylim(-4, 4)

axs[2].scatter(x_generated[:, 0], x_generated[:, 1], s=5, color='green')
axs[2].set_title('Generated Samples from Flow')

plt.show()

## 4. Advanced Architectures: MAF and NSF

RealNVP is simple but has limited expressivity because the Jacobian is constrained to be triangular in a specific way (coupling). We will now explore two more advanced architectures: **Masked Autoregressive Flow (MAF)** and **Neural Spline Flow (NSF)**.

### Masked Autoregressive Flow (MAF)

MAF uses the autoregressive property:
$$ p(x) = \prod_{i=1}^D p(x_i | x_{1:i-1}) $$

Each dimension $x_i$ is transformed based on previous dimensions. We can implement this efficiently using a **Masked Autoencoder (MADE)**, which masks the weights of a standard MLP to respect the autoregressive order.

The transformation is affine (like RealNVP), but the parameters depend on all previous variables:
$$ y_i = x_i \cdot \exp(s(x_{1:i-1})) + t(x_{1:i-1}) $$

In [ ]:
# Masked Linear Layer
class MaskedLinear(nn.Linear):
    def __init__(self, in_features, out_features, bias=True, mask=None):
        super().__init__(in_features, out_features, bias)
        self.register_buffer('mask', mask)
    
    def forward(self, input):
        return nn.functional.linear(input, self.weight * self.mask, self.bias)

# MADE Network
class MADE(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim_multiplier=2):
        super().__init__()
        self.input_dim = input_dim
        
        # Create masks
        # m[l] gives the max input index that the unit depends on
        # Input layer: 1, 2, ..., D
        m_input = np.arange(input_dim)
        
        # Hidden layers: can depend on input indices
        m_hidden_1 = np.random.randint(0, input_dim - 1, hidden_dim)
        m_hidden_2 = np.random.randint(0, input_dim - 1, hidden_dim)
        
        # Masks construction
        # M_1: hidden_1 vs input
        mask1 = torch.from_numpy(m_hidden_1[:, None] >= m_input[None, :]).float()
        # M_2: hidden_2 vs hidden_1
        mask2 = torch.from_numpy(m_hidden_2[:, None] >= m_hidden_1[None, :]).float()
        # M_out: output vs hidden_2
        # Output k depends on input 1...k-1. So unit k in output must depend on hidden units with index < k
        # We have multiple outputs per dimension (e.g. scale and shift)
        m_output = np.repeat(np.arange(input_dim), output_dim_multiplier)
        mask3 = torch.from_numpy(m_output[:, None] > m_hidden_2[None, :]).float()
        
        self.net = nn.Sequential(
            MaskedLinear(input_dim, hidden_dim, mask=mask1),
            nn.ReLU(),
            MaskedLinear(hidden_dim, hidden_dim, mask=mask2),
            nn.ReLU(),
            MaskedLinear(hidden_dim, input_dim * output_dim_multiplier, mask=mask3)
        )

    def forward(self, x):
        return self.net(x)

class MAF(nn.Module):
    def __init__(self, input_dim, hidden_dim, n_layers=4):
        super().__init__()
        self.layers = nn.ModuleList()
        for _ in range(n_layers):
            self.layers.append(MADE(input_dim, hidden_dim, output_dim_multiplier=2))
            
    def forward(self, x):
        # Forward in MAF is usually defined as the inverse in RealNVP notation (z = f(x))
        # Here we follow sbi/iaf convention: forward is x -> z (normalizing)
        log_det_sum = 0
        for made in self.layers:
            # MADE outputs parameters for each dimension based on previous dimensions
            out = made(x)
            mu, log_alpha = out.chunk(2, dim=1)
            
            # y = (x - mu) / exp(log_alpha)  [Inverse Autoregressive]
            # But MAF is autoregressive in forward pass: 
            # y_i = (x_i - mu_i(x_{<i})) / exp(alpha_i(x_{<i}))
            # This is fast to compute.
            
            # To match RealNVP implementation (forward x->z), we do:
            # z = x * exp(alpha) + mu
            # But wait, MAF's fast pass is x -> z? 
            # Actually: MAF is fast for Likelihood estimation (x -> z).
            # IAF is fast for Sampling (z -> x).
            # Since we want to train density estimation, we use MAF direction.
            
            # x = z * exp(s) + t  <= This is slow for MAF because t depends on x
            # z = (x - t) * exp(-s) <= This is fast for MAF.
            
            # So: z = (x - mu) * exp(-alpha)
            
            x = (x - mu) * torch.exp(-log_alpha)
            log_det_sum += -torch.sum(log_alpha, dim=1)
            
            # Flip for mixing dimensions between layers
            x = x.flip(dims=[1])
            
        return x, log_det_sum

    # Inverse (Sampling) is slow for MAF (sequential)
    def inverse(self, z):
        for made in reversed(self.layers):
            z = z.flip(dims=[1])
            # We need to solve x_i one by one
            x = torch.zeros_like(z)
            for i in range(z.shape[1]):
                out = made(x)
                mu, log_alpha = out.chunk(2, dim=1)
                # z_i = (x_i - mu_i) * exp(-alpha_i)
                # x_i = z_i * exp(alpha_i) + mu_i
                x[:, i] = z[:, i] * torch.exp(log_alpha[:, i]) + mu[:, i]
        return x

### Neural Spline Flow (NSF)

Affine transformations ($y = ax+b$) are limited. **Spline Flows** use a piecewise polynomial function.

A **Rational Quadratic Spline (RQS)** consists of bins. In each bin, the function is the ratio of two quadratic polynomials. It is monotonic and differentiable.

We will implement a simplified version with a `CouplingLayer` where the transformation is a spline instead of affine.

In [ ]:
# Simplified Rational Quadratic Spline Helper
def r_q_spline(inputs, unnormalized_widths, unnormalized_heights, unnormalized_derivatives, inverse=False):
    # Inputs: [batch, 1] (we apply elementwise)
    # Params: [batch, K] (K bins)
    
    # 1. Softmax to get widths/heights (must be positive)
    w = torch.softmax(unnormalized_widths, dim=1)
    h = torch.softmax(unnormalized_heights, dim=1)
    #d = torch.nn.functional.softplus(unnormalized_derivatives) # Derivatives must be > 0
    
    # 2. Cumulative widths/heights to get knots
    cum_w = torch.cumsum(w, dim=1)
    cum_w = torch.nn.functional.pad(cum_w, (1, 0), mode='constant', value=0.0)
    cum_h = torch.cumsum(h, dim=1)
    cum_h = torch.nn.functional.pad(cum_h, (1, 0), mode='constant', value=0.0)
    
    # Domain is usually [-B, B]. Let's map inputs to [0, 1] for simplicity or assume inputs are within range.
    # For simplicity here, we assume inputs are inside the range [0, 1] mapped from [-2.5, 2.5]
    # (A full implementation handles out-of-bounds with linear tails)
    
    # ... (Implementing full RQS math is very verbose. We will implement a simplified piecewise linear approx for demo purposes 
    # or a very basic monotonic transform to show the concept of non-affine).
    
    # Let's implement a Piecewise Linear Spline for simplicity and stability in a scratch tutorial.
    # It's easier to code and demonstrates the "binned" flexibility.
    
    # Find bin index
    if not inverse:
        # forward: x -> y
        # Map x from [-3, 3] to [0, 1]
        x_01 = (inputs + 3) / 6.0
        x_01 = torch.clamp(x_01, 0, 0.999)
        
        bin_idx = torch.searchsorted(cum_w, x_01) - 1
        bin_idx = bin_idx.clamp(0, w.shape[1]-1)
        
        # Gather bin params
        w_b = torch.gather(w, 1, bin_idx)
        h_b = torch.gather(h, 1, bin_idx)
        cum_w_b = torch.gather(cum_w, 1, bin_idx)
        cum_h_b = torch.gather(cum_h, 1, bin_idx)
        
        # Relative position in bin
        xi = (x_01 - cum_w_b) / w_b
        
        # Transformation (Linear in bin)
        y_01 = cum_h_b + xi * h_b
        
        # Map back
        y = y_01 * 6.0 - 3.0
        
        # Jacobian: dy/dx = (dy/d_01) * (d_01/dx) = (h/w) * 1
        log_det = torch.log(h_b) - torch.log(w_b)
        
        return y, log_det
        
    else:
        # inverse: y -> x
        y_01 = (inputs + 3) / 6.0
        y_01 = torch.clamp(y_01, 0, 0.999)
        
        bin_idx = torch.searchsorted(cum_h, y_01) - 1
        bin_idx = bin_idx.clamp(0, h.shape[1]-1)
        
        w_b = torch.gather(w, 1, bin_idx)
        h_b = torch.gather(h, 1, bin_idx)
        cum_w_b = torch.gather(cum_w, 1, bin_idx)
        cum_h_b = torch.gather(cum_h, 1, bin_idx)
        
        yi = (y_01 - cum_h_b) / h_b
        x_01 = cum_w_b + yi * w_b
        
        x = x_01 * 6.0 - 3.0
        return x

class SplineCouplingLayer(nn.Module):
    def __init__(self, input_dim, hidden_dim, mask, n_bins=8):
        super().__init__()
        self.mask = mask
        self.n_bins = n_bins
        
        # Predict bin widths and heights (and derivatives for RQS, here just W and H for Linear)
        # We need (W, H) for each bin for each dimension
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, input_dim * n_bins * 2) # Widths and Heights
        )

    def forward(self, x):
        x_masked = x * self.mask
        params = self.net(x_masked).view(x.shape[0], x.shape[1], self.n_bins * 2)
        
        # Split params
        unnorm_w = params[..., :self.n_bins]
        unnorm_h = params[..., self.n_bins:]
        
        # Apply spline to x where mask is 0
        # To make it vectorized, we calculate for all, then mix.
        # Note: We treat each dim independently given the params
        
        # Flatten for helper
        x_flat = x.view(-1, 1)
        w_flat = unnorm_w.reshape(-1, self.n_bins)
        h_flat = unnorm_h.reshape(-1, self.n_bins)
        dummy_d = torch.zeros_like(w_flat) # Placeholder
        
        y_flat, log_det_flat = r_q_spline(x_flat, w_flat, h_flat, dummy_d)
        
        y_full = y_flat.view(x.shape)
        log_det_full = log_det_flat.view(x.shape)
        
        # Apply mask
        y = x * self.mask + y_full * (1 - self.mask)
        log_det = torch.sum(log_det_full * (1 - self.mask), dim=1)
        
        return y, log_det

    def inverse(self, y):
        y_masked = y * self.mask
        params = self.net(y_masked).view(y.shape[0], y.shape[1], self.n_bins * 2)
        
        unnorm_w = params[..., :self.n_bins]
        unnorm_h = params[..., self.n_bins:]
        
        y_flat = y.view(-1, 1)
        w_flat = unnorm_w.reshape(-1, self.n_bins)
        h_flat = unnorm_h.reshape(-1, self.n_bins)
        dummy_d = torch.zeros_like(w_flat)
        
        x_flat = r_q_spline(y_flat, w_flat, h_flat, dummy_d, inverse=True)
        x_full = x_flat.view(y.shape)
        
        x = y * self.mask + x_full * (1 - self.mask)
        return x

class NSF(nn.Module):
    def __init__(self, input_dim, hidden_dim, n_layers=4):
        super().__init__()
        self.layers = nn.ModuleList()
        mask = torch.tensor([1, 0], dtype=torch.float32)
        for _ in range(n_layers):
            self.layers.append(SplineCouplingLayer(input_dim, hidden_dim, mask))
            mask = 1 - mask
            
    def forward(self, x):
        log_det_sum = 0
        for layer in self.layers:
            x, log_det = layer(x)
            log_det_sum += log_det
        return x, log_det_sum
        
    def inverse(self, z):
        for layer in reversed(self.layers):
            z = layer.inverse(z)
        return z

## Comparison Training

We will now train RealNVP, MAF, and NSF side-by-side.

In [ ]:
models = {
    'RealNVP': RealNVP(2, 64, n_layers=6),
    'MAF': MAF(2, 64, n_layers=4),
    'NSF (Linear Spline)': NSF(2, 64, n_layers=6)
}

results = {}

for name, model in models.items():
    print(f"Training {name}...")
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    loss_hist = []
    
    for epoch in range(400):
        optimizer.zero_grad()
        z, log_det = model(X_train)
        log_prob_z = prior.log_prob(z)
        loss = -torch.mean(log_prob_z + log_det)
        loss.backward()
        optimizer.step()
        loss_hist.append(loss.item())
    
    results[name] = {'model': model, 'loss': loss_hist}
    print(f"Final Loss: {loss_hist[-1]:.4f}")

# Visualization
fig, axs = plt.subplots(2, 3, figsize=(15, 10))
z_sample = prior.sample((2000,))

for i, (name, res) in enumerate(results.items()):
    # Plot Loss
    axs[0, i].plot(res['loss'])
    axs[0, i].set_title(f'{name} Loss')
    
    # Plot Samples
    with torch.no_grad():
        if name == 'MAF':
            # MAF inverse is slow, show a smaller subset if needed, or wait
            x_gen = res['model'].inverse(z_sample[:500]).numpy()
        else:
            x_gen = res['model'].inverse(z_sample).numpy()
            
    axs[1, i].scatter(x_gen[:, 0], x_gen[:, 1], s=5, color='green')
    axs[1, i].set_title(f'{name} Samples')
    axs[1, i].set_xlim(-1.5, 2.5)
    axs[1, i].set_ylim(-1, 1.5)

plt.tight_layout()
plt.show()